# Steps:
- ~~import data~~
- ~~select only the proper sequences -> dataframe~~
- ~~filter the sequences portion of interest (remove bla bla.)~~
- ~~create the fasta/txt file with the > inserction // name ++ sequence~~
- ~~run the file~~
- filter Results: MANTAIN only result with OK bp and ASSOTIATED seq_id
- inspect and CHECK the results
- correlate the results with data
- (optional) intersect the domain and find the sequences

- ---------> data analisys using these BP classification

In [1]:
# Standard libraries used throughout the notebook
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import seaborn as sns
import difflib
import os


# --------------------------------------------------------------------------
# PARAMETER — PATH_DATASET
# --------------------------------------------------------------------------

PATH_DATASET = r"C:\Users\marco\Desktop\Tesi\.LabWork\Data\Data from Suzanne\fitness_calc_PB_H3B.xlsx"

# We load the raw file with no changes.
# data_raw will never be modified — it stays as the original reference.
data_raw = pd.read_excel(PATH_DATASET)


# --------------------------------------------------------------------------
# Data Filtering
# --------------------------------------------------------------------------

MIN_READS_THRESHOLD = 50

n_total   = len(data_raw)
mask_ok   = data_raw["Input_counts"] >= MIN_READS_THRESHOLD
n_removed = (~mask_ok).sum()
n_kept    = mask_ok.sum()

data_filtered = data_raw[mask_ok].copy().reset_index(drop=True)


### Create useful functions

In [2]:
def filter_sequence(sequence: str, to_find: str) -> str:
    """Optimizes the input sequence for the SVM finder by removing the final

    section that follows the specified sub-string (e.g., 'CAG').

    Parameters:
        - sequence: str, the string to be cut
        - to_find: str, the sub-string to locate

    Returns:
        - str: The subsetted string, including the 'to_find' sequence at the
        end.
    Raise:
        - ValueError if the to_find string isn't in the sequence
    """
    # Find the index of the last occurrence (searching from right to left)
    idx = sequence.rfind(to_find)

    # If the sub-string is not found, rfind returns -1.
    if idx == -1:
    # In this case, we return the original sequence.
        raise ValueError(f"The string {to_find} is not in {sequence}.")

    # Slice the string to keep everything from the beginning
    # up to the end of 'to_find' (inclusive)
    return sequence[: idx + len(to_find)]

def BP_extractor(sequence: str):
    '''Given the Experimental design, retrieve the BP part from the 
    input sequence. It is known that the BP_sequence is between
    28th and 21th nt starting from the end (starts from the end
    because the ANCHOR preceed the BP and has variable lenght)    
    '''
    return sequence[-28:-21]

def translator_to_DNA(tuples):
    nuove_tuple = []
    for nome, seq in tuples:
        # Trasforma in maiuscolo e sostituisce la U con la T
        seq_dna = seq.upper().replace('U', 'T')
        # Salva la coppia nella nuova lista
        nuove_tuple.append((nome, seq_dna))
    return nuove_tuple

### Manipulate data

In [4]:
df = pd.DataFrame(data_filtered.loc[:, ['nt_seq', 'full_seq','ANCHOR','DECOY','PYT','seq_id']].copy())
mask = df['nt_seq'].notna() & df['full_seq'].notna()
dna = pd.DataFrame(df.loc[mask, :].copy())



dna["seq_feature"] = (
    dna["DECOY"].astype(str)
    + '__'
    + "PYT "
    + dna["PYT"].astype(str)
    + '__'
    + "ANCHOR "
    + dna["ANCHOR"].astype(str)
    + '__'
    + 'Seq_id: ' 
    + dna['seq_id'].astype(str)
)

### Sequence inspection

In [5]:
ref = 'cgTAATACGACTCACTATAGGGcUACUACGUCAGCUCGUCUCGAGGGUACUAACgcgUUUUUCUUUUCUUUUCAGGGUACGCAU'

tuples = [
    ('t7', 'cgTAATACGACTCACTATAGGGc'),
    ('decoy', 'UACUAC'),
    ('ANCHOR_1', 'GGGUUUCCUGAAGCUUUCG'),
    ('ANCHOR_2', 'GUCAGCUCGUCUCGAGGG'),
    ('branch_point', 'UACUAAC'),
    ('nt_linker', 'gcg'),
    ('strong_PYT', 'UUUUUCUUUUCUUUU'),
    ('weak_PYT', 'UUgaCUUgUUgaCCU'),
    ('filtered_junction', 'CAG')
]

tuples = translator_to_DNA(tuples)

lunghezze = dna['full_seq'].str.len()

# 2. Verifica se il numero di lunghezze uniche è pari a 1
if lunghezze.nunique() == 1:
    print(f"Sì, tutte le stringhe hanno la stessa lunghezza ({lunghezze.iloc[0]} nucleotidi).")
else:
    print("No, ci sono stringhe con lunghezze diverse.")
    print(lunghezze.nunique())


print()
for name, seq in tuples:
    print(f"The length of {name} is: {len(seq)}")
print()
for name, seq in tuples:
    check = dna['full_seq'].str.contains(seq, case=False, na=False).sum() / len(dna['full_seq'])
    print(f" {round((check*100), 0)}% of sequences with the feature: {name}")

No, ci sono stringhe con lunghezze diverse.
2

The length of t7 is: 23
The length of decoy is: 6
The length of ANCHOR_1 is: 19
The length of ANCHOR_2 is: 18
The length of branch_point is: 7
The length of nt_linker is: 3
The length of strong_PYT is: 15
The length of weak_PYT is: 15
The length of filtered_junction is: 3

 100.0% of sequences with the feature: t7
 16.0% of sequences with the feature: decoy
 50.0% of sequences with the feature: ANCHOR_1
 50.0% of sequences with the feature: ANCHOR_2
 0.0% of sequences with the feature: branch_point
 100.0% of sequences with the feature: nt_linker
 53.0% of sequences with the feature: strong_PYT
 47.0% of sequences with the feature: weak_PYT
 100.0% of sequences with the feature: filtered_junction


### Function Tester

In [ ]:
seq_raw = dna.loc[0]['full_seq'] 
seq_filtered = filter_sequence(dna.loc[0]['full_seq'], 'CAG')

# Create the difflib object
d = difflib.Differ()
diff = d.compare(seq_raw, seq_filtered)
# Print the result
print("Difference between the 2 strings")
print("".join(diff))
print()



sample_seq = dna['full_seq'][0]
BP = BP_extractor(sample_seq)
pb_extracted = len(BP)
if pb_extracted == 7:
    d = difflib.Differ()
    diff = d.compare(sample_seq, BP)
    # Print the result
    print("Difference between the 2 strings")
    print("".join(diff))
    print()

else: print(f"adjust the function, the len is wrong: {pb_extracted}")

Difference between the 2 strings
  C  G  T  A  A  T  A  C  G  A  C  T  C  A  C  T  A  T  A  G  G  G  C  G  T  C  G  T  C  G  G  G  T  T  T  C  C  T  G  A  A  G  C  T  T  T  C  G  A  A  A  A  A  A  A  G  C  G  T  T  G  A  C  T  T  G  T  T  G  A  C  C  T  C  A  G- G- G- T- A- C- G- C- A- T

The function works
Difference between the 2 strings
- C- G- T- A- A- T- A- C- G- A- C- T- C- A- C- T- A- T- A- G- G- G- C- G- T- C- G- T- C- G- G- G- T- T- T- C- C- T- G- A- A- G- C- T- T- T- C- G- A- A- A- A- A- A- A- G- C  G  T  T  G  A  C  T- T- G- T- T- G- A- C- C- T- C- A- G- G- G- T- A- C- G- C- A- T



### Filter the dataset

In [7]:
pattern = 'CAG'

dna['full_seq'] = dna['full_seq'].apply(
    lambda seq: filter_sequence(seq, pattern)
)

dna['BP_extracted'] = dna['full_seq'].apply(
    lambda s: BP_extractor(s)
)

### Create the fasta file

In [8]:
col_id = "seq_feature"
col_seq = "full_seq"

file_name = "test_sequence.fasta"

target_folder = r"C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M"
full_path = os.path.join(target_folder, file_name)

with open(full_path, "w") as fasta_file:
    for row in dna.itertuples(index=False):
        header = getattr(row, col_id)
        sequence = getattr(row, col_seq)
        
        fasta_file.write(f">{header}\n{sequence}\n")

print(f"FASTA file directly created in: {full_path}")

FASTA file directly created in: C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M\test_sequence.fasta


### !!!Simulation Runned on terminal (code: python3 svm_bpfinder.py -i test_sequence.fasta -s Hsap -l 100 -d 10 > BP_results.txt)!!!

### Result Filtering

In [36]:
PATH = r"C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M\BP_results.txt"

df = pd.read_csv(PATH, sep="\t")
res = df.loc[:, ['seq_id', 'bp_seq','svm_scr']]
res['bp_seq']= res['bp_seq'].apply(
    lambda s: s.upper()
) 
res['seq_id_num'] = res['seq_id'].str.extract(r'Seq_id:\s*(\d+)').astype(int)

### Mantain only result relative to KNOWN BP (present in the experimental design)

In [20]:
BP_list = dna['BP_extracted'].dropna().unique()
pattern_regex = '|'.join(BP_list)
mask = res['bp_seq'].str.contains(pattern_regex, case=False, na=False)
print(f"Filtering succesfully executed!!\nThere are {(sum(mask)/ len(res['bp_seq']))*100}% of experimental design BP")

Filtering succesfully executed!!
There are 100.0% of experimental design BP


# make sense more sequences here than in the input bp????

In [ ]:
res = res[mask].reset_index() 

### Result Analysis

We obtain a file with these information:
- seq_id  
- agez: Lunghezza della Zona di Esclusione dei dinucleotidi AG ($AGEZ$). Misura la regione a monte del sito di splicing in cui l'assenza di altri AG previene la competizione del segnale.  
- ss_dist: Distanza in nucleotidi tra la 'A' del Branch Point valutato e il sito di splicing al 3' ($3\text{SS}$). 
- bp seq: La sequenza locale del Branch Point (un nonamero di 9 basi, da $-5$ a $+3$ rispetto alla Adenina centrale).  
- bp_scr: Il punteggio intrinseco della sequenza del nonamero, calcolato con il modello di Markov. Valuta quanto la sequenza somigli a un vero consenso biologico umano.  
- y cont: Contenuto (densità) di pirimidine (C e T/U) tra la Adenina del BP e il sito di splicing al 3'.  
- ppt_off: Distanza (offset) del Polipirimidine Tract rispetto alla Adenina del BP.  
- ppt_len: Lunghezza complessiva del Polipirimidine Tract identificato a valle.  
- ppt_scr: Punteggio di forza del Polipirimidine Tract. Più è alto, più il tratto è ricco di timine/uracili.
- svm_scr: Il punteggio finale globale del Branch Point, calcolato dall'algoritmo di Machine Learning (SVM) combinando i tratti precedenti.

### Sequence Extraction

In [30]:
col = 'svm_scr'
q1 = res[col].quantile(0.25)
q2 = res[col].quantile(0.50)  # Median
q3 = res[col].quantile(0.75)

q1_df = res[res[col] < q1]
q2_df = res[res[col] < q2]
q3_df = res[res[col] < q2]

q1_df

,level_0,index,seq_id,bp_seq,svm_scr
0,0,0,Decoy 7__PYT Weak__ANCHOR 1__Seq_id: 110593,GACTCACTA,-2.656335
3,3,3,Decoy 7__PYT Strong__ANCHOR 1__Seq_id: 102401,GACTCACTA,-1.306384
5,5,5,Decoy 7__PYT Weak__ANCHOR 1__Seq_id: 110596,GACTCACTA,-2.650766
10,10,10,Decoy 7__PYT Weak__ANCHOR 1__Seq_id: 110595,GACTCACTA,-2.656335
13,13,13,Decoy 7__PYT Strong__ANCHOR 1__Seq_id: 102403,GACTCACTA,-1.306384
...,...,...,...,...,...
313163,313163,313163,Decoy 6__PYT Weak__ANCHOR 2__Seq_id: 91482,GACTCACTA,-2.578871
313164,313164,313164,Decoy 6__PYT Weak__ANCHOR 2__Seq_id: 91482,CGGTCAGCT,-1.912629
313172,313172,313172,Decoy 6__PYT Weak__ANCHOR 2__Seq_id: 91477,TTTTTAAGC,-2.117866
313188,313188,313188,Decoy 6__PYT Weak__ANCHOR 2__Seq_id: 91479,TTTTTAGGC,-2.279228
